In [23]:
import csv

with open("books.csv") as f:
    books = list(csv.DictReader(f))

books

[{'title': 'Think Python 2e',
  'book_url': 'https://greenteapress.com/wp/think-python-2e/',
  'pdf_url': 'https://greenteapress.com/thinkpython2/thinkpython2.pdf'},
 {'title': 'Think DSP',
  'book_url': 'https://greenteapress.com/wp/think-dsp/',
  'pdf_url': 'https://greenteapress.com/thinkdsp/thinkdsp.pdf'},
 {'title': 'Think Complexity 2e',
  'book_url': 'https://greenteapress.com/wp/think-complexity/',
  'pdf_url': 'https://greenteapress.com/complexity2/thinkcomplexity2.pdf'},
 {'title': 'Think Java 2e',
  'book_url': 'https://greenteapress.com/wp/think-java-2e/',
  'pdf_url': 'https://greenteapress.com/thinkjava7/thinkjava2.pdf'},
 {'title': 'Physical Modeling in MATLAB',
  'book_url': 'https://greenteapress.com/wp/physical-modeling-in-matlab/',
  'pdf_url': 'https://github.com/AllenDowney/PhysicalModelingInMatlab/raw/master/PhysicalModelingInMatlab4.pdf'},
 {'title': 'Think OS',
  'book_url': 'https://greenteapress.com/wp/think-os/',
  'pdf_url': 'https://greenteapress.com/thinko

In [24]:
from pathlib import Path

books_dir = Path("books")
books_dir.mkdir(exist_ok=True)

In [25]:
import requests

for book in books:
    pdf_url = book["pdf_url"]
    filename = pdf_url.split("/")[-1]
    
    response = requests.get(pdf_url)
    (books_dir / filename).write_bytes(response.content)
    
    print(filename)

thinkpython2.pdf
thinkdsp.pdf
thinkcomplexity2.pdf
thinkjava2.pdf
PhysicalModelingInMatlab4.pdf
thinkos.pdf
Think-C.pdf


In [26]:
from pathlib import Path

books_text_dir = Path("books_text")

documents = []

for path in books_text_dir.glob("*.md"):
    text = path.read_text()
    lines = [line for line in text.splitlines() if line.strip()]

    documents.append({
        "source": path.name,
        "content": lines
    })

In [27]:
len(documents)

7

In [28]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=100,
    step=50
)

In [29]:
think_python_chunks = [
    chunk for chunk in chunks
    if chunk["source"] == "thinkpython2.md"
]

len(think_python_chunks)

189

In [30]:
def prepare_documents(chunks):
    documents = []

    for chunk in chunks:
        document = {
            "source": chunk["source"],
            "content": "\n".join(chunk["content"])
        }
        documents.append(document)

    return documents

In [31]:
documents_for_index = prepare_documents(chunks)

In [32]:
from minsearch import Index

index = Index(text_fields=["content"])
index.fit(documents_for_index)

In [33]:
len(documents_for_index)

988

In [34]:
results = index.search(
    "python function definition",
    num_results=5
)

results

[{'source': 'PhysicalModelingInMatlab4.md',
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables

In [35]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

In [36]:
from openai import OpenAI

openai_client = OpenAI()

def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [37]:
def search(question):
    return index.search(question, num_results=5)

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [38]:
answer = rag("python function definition")
answer

"In Python, a function is defined using the `def` keyword, followed by the function name and parentheses containing any parameters. Here's a simple example of a function definition:\n\n```python\ndef my_function(x):\n    result = x ** 2  # Compute the square of the input\n    return result    # Return the result\n```\n\n### Key Points:\n- **Function Name**: The name after `def` (e.g., `my_function`) is how you will call the function.\n- **Parameters**: The variable inside parentheses (e.g., `x`) allows the function to take input.\n- **Return Statement**: The `return` statement sends a value back to where the function was called.\n\nYou can call the function like this:\n\n```python\noutput = my_function(3)  # This will return 9, as 3 squared is 9.\n```\n\nThis encapsulates code, promotes reusability, and maintains a clean namespace, reducing the chances of variable name collisions. Functions can also take multiple parameters, as shown in the detailed examples related to MATLAB functions

In [39]:
def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

In [40]:
result = rag("python function definition")
result

{'answer': "A Python function is defined using the `def` keyword, followed by the function name and parentheses containing any parameters. The function body is indented and contains the execution logic. Here's a basic structure of a Python function:\n\n```python\ndef function_name(parameters):\n    # Function body\n    # Perform tasks\n    return output_value  # optional\n```\n\nFor example, a simple function that adds two numbers might look like this:\n\n```python\ndef add_numbers(a, b):\n    return a + b\n```\n\nIn this function:\n\n- `add_numbers` is the name of the function.\n- `a` and `b` are parameters (inputs).\n- The `return` statement sends back the result of `a + b`.\n\nYou can call the function by using its name and passing arguments:\n\n```python\nresult = add_numbers(5, 3)\nprint(result)  # Output will be 8\n```\n\nThis structure allows for modular code, enabling you to define a block of code once and reuse it throughout your program.",
 'input_tokens': 7888,
 'output_toke

In [41]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(
        description="The main answer to the user's question in markdown"
    )
    found_answer: bool = Field(
        description="True if relevant information was found in the documentation"
    )
    confidence: float = Field(
        description="Confidence score from 0.0 to 1.0"
    )
    confidence_explanation: str = Field(
        description="Explanation about the confidence level"
    )
    answer_type: Literal[
        "how-to",
        "explanation",
        "troubleshooting",
        "comparison",
        "reference"
    ] = Field(
        description="The category of the answer"
    )
    followup_questions: list[str] = Field(
        description="Suggested follow-up questions"
    )

In [42]:
def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return {
        "answer": response.output_parsed,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

In [43]:
result_structured = rag("python function definition")
result_structured

{'answer': RAGResponse(answer='### Python Function Definition\n\nA function in Python is defined using the `def` keyword, followed by the function name and parentheses containing optional parameters. Here’s a basic structure for defining a function:\n\n```python\ndef function_name(parameters):\n    """\n    Optional documentation string.\n    """\n    # Function body - the code that runs when the function is called\n    # ...\n    return output\n```\n\n#### Example\nHere’s an example of a simple function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    """\n    Returns the sum of a and b.\n    """\n    return a + b\n```\n\nIn this example:\n- `add_numbers` is the function name.\n- `a` and `b` are parameters.\n- The `return` statement sends back the function\'s result.\n\n### Important Notes:\n- Function names should be descriptive to indicate what the function does.\n- You can call the function like this: `result = add_numbers(5, 10)`.\n- Functions can return multiple v